<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/16%20-%20Problemas%20Eulerianos%20e%20Inspe%C3%A7%C3%A3o%20de%20Infraestrutura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção Autônoma da Infraestrutura da Linha de Envase

Neste notebook modelamos a malha física sanitária de tubulações da **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)** e implementamos o **Algoritmo de Hierholzer** para gerar a rota ótima de varredura e inspeção de corrosão/espessura pelo robô industrial autônomo sem repetição de dutos.


In [1]:
from collections import defaultdict
from typing import List, Tuple, Dict, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[Any]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "INF" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)
        self.dutos_info = {}

    def adicionar_duto(self, u: str, v: str, id_duto: str, comprimento_m: float = 5.0, tag_valvula: str = "SAN_PIPE"):
        # Grafo nao-dirigido: o robo de inspecao trafega em ambos os sentidos da tubulacao
        self.adj[u].append((v, id_duto))
        self.adj[v].append((u, id_duto))
        self.dutos_info[id_duto] = {
            "ID_Duto": id_duto, "Segmento": f"{u} <-> {v}",
            "Comprimento (m)": comprimento_m, "Tag_ISA": tag_valvula
        }

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for v in sorted(self.adj.keys()):
            grau = len(self.adj[v])
            paridade = "PAR (Euleriano)" if grau % 2 == 0 else "IMPAR (Violacao)"
            graus.append({"Componente / Vertice": v, "Grau deg(v)": grau, "Classificacao": paridade})
        return graus

    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        eul, imp = self.verificar_euleriano()
        if not eul:
            raise ValueError(f"O grafo nao e Euleriano! Vertices impares: {imp}")

        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []

        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())

        circuito.reverse()
        return circuito

# Construcao da malha sanitaria de inspecao da Linha de Envase de Bebidas
g_insp = GrafoInspecaoEuleriano()

# Anel externo e acesso da Estacao de Manutencao do Robo
g_insp.adicionar_duto("Base_Manutencao", "TS1_Suprimento", "d01_base_ts1", 6.0, "ENGATE_RAPIDO_1")
g_insp.adicionar_duto("Base_Manutencao", "EST_Envase", "d02_est_base", 4.0, "ENGATE_RAPIDO_2")

# Malha hidraulica de succao e recalque
g_insp.adicionar_duto("TS1_Suprimento", "VS1_Succao", "d03_ts1_vs1", 5.0, "VS1")
g_insp.adicionar_duto("VS1_Succao", "BC1_Bomba", "d04_vs1_bc1", 3.0, "SP1_SQ1")
g_insp.adicionar_duto("BC1_Bomba", "AS1_Acumulador", "d05_bc1_as1", 8.0, "CHECK_V")

# Linha de Alivio e Reciclo de Seguranca
g_insp.adicionar_duto("AS1_Acumulador", "VALV_Alivio", "d06_as1_valv", 6.0, "SP2_VS3")
g_insp.adicionar_duto("VALV_Alivio", "TS1_Suprimento", "d07_valv_ts1", 12.0, "RET_V")

# Linhas de Dosagem, Bypass e Bico de Envase
g_insp.adicionar_duto("AS1_Acumulador", "VS2_Envase", "d08_as1_vs2", 10.0, "VS2")
g_insp.adicionar_duto("VS2_Envase", "SQ2_Medicao", "d09_vs2_sq2", 2.0, "SQ2_IN")
g_insp.adicionar_duto("AS1_Acumulador", "SQ2_Medicao", "d10_as1_bypass", 14.0, "XV_BYPASS")
g_insp.adicionar_duto("SQ2_Medicao", "EST_Envase", "d11_sq2_est", 1.5, "BICO_FILL")

# Linhas de interligacao para balanceamento euleriano dos graus (Tubos de Flush/CIP)
g_insp.adicionar_duto("TS1_Suprimento", "BC1_Bomba", "d12_cip_recirc", 7.0, "VALV_CIP_1")
g_insp.adicionar_duto("BC1_Bomba", "SQ2_Medicao", "d13_cip_flush", 9.0, "VALV_CIP_2")

print("=== AUDITORIA TOPOLOGICA DE GRAUS (TEOREMA DE EULER) ===")
print(formatar_tabela(g_insp.obter_graus()))

eul, imp = g_insp.verificar_euleriano()
print(f"\nGrafo e estritamente Euleriano: {eul} | Vertices com grau impar: {imp}")
assert eul is True, "Erro: O grafo deve ter todos os graus pares para ser Euleriano!"

# Calculo da Rota Otima de Inspecao de Corrosao via Algoritmo de Hierholzer
rota_robo = g_insp.calcular_circuito_hierholzer("Base_Manutencao")
print("\n=== CIRCUITO EULERIANO DE INSPECAO DE TUBULACOES (HIERHOLZER) ===")
print(" -> ".join(rota_robo))

# Metricas da Missao de Manutencao
num_visitas_arestas = len(rota_robo) - 1
total_arestas_grafo = len(g_insp.dutos_info)
comprimento_total_inspecao = sum(d["Comprimento (m)"] for d in g_insp.dutos_info.values())

print(f"\nTotal de Dutos Inspecionados: {total_arestas_grafo}")
print(f"Passos do Circuito do Robo: {num_visitas_arestas}")
print(f"Comprimento Total do Percurso: {comprimento_total_inspecao:.1f} m")

# Assercoes de conformidade matematica estrita
assert rota_robo[0] == "Base_Manutencao"
assert rota_robo[-1] == "Base_Manutencao"
assert num_visitas_arestas == total_arestas_grafo, "O robo deve percorrer cada duto exatamente uma vez!"
print("\n✓ Circuito Euleriano validado: cada tubulacao do skid foi inspecionada exatamente UMA vez sem redundancia!")


=== AUDITORIA TOPOLOGICA DE GRAUS (TEOREMA DE EULER) ===
Componente / Vertice | Grau deg(v) | Classificacao  
---------------------+-------------+----------------
AS1_Acumulador       | 4           | PAR (Euleriano)
BC1_Bomba            | 4           | PAR (Euleriano)
Base_Manutencao      | 2           | PAR (Euleriano)
EST_Envase           | 2           | PAR (Euleriano)
SQ2_Medicao          | 4           | PAR (Euleriano)
TS1_Suprimento       | 4           | PAR (Euleriano)
VALV_Alivio          | 2           | PAR (Euleriano)
VS1_Succao           | 2           | PAR (Euleriano)
VS2_Envase           | 2           | PAR (Euleriano)

Grafo e estritamente Euleriano: True | Vertices com grau impar: []

=== CIRCUITO EULERIANO DE INSPECAO DE TUBULACOES (HIERHOLZER) ===
Base_Manutencao -> EST_Envase -> SQ2_Medicao -> BC1_Bomba -> TS1_Suprimento -> VALV_Alivio -> AS1_Acumulador -> SQ2_Medicao -> VS2_Envase -> AS1_Acumulador -> BC1_Bomba -> VS1_Succao -> TS1_Suprimento -> Base_Manutencao

Tota